# **Build the fist MIL pipeline : 1D CNN + Classifier**

## **Defining the training on the GPU**

In [1]:
from pathlib import Path
CHECKPOINT_DIR = Path("checkpoints_mil_classifier_3s")
import mlflow
from Tracking.mlflow_config import *

c:\Users\aliai\Desktop\M2 I2A\Optimisation des modeles - Ouzar\Projet\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
experiment_name = "MIL_Model_3s_"

In [3]:
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
print("Dossier des checkpoints :", CHECKPOINT_DIR.resolve())

Dossier des checkpoints : C:\Users\aliai\Desktop\M2 I2A\Optimisation des modeles - Ouzar\Projet\Base_line\checkpoints_mil_classifier_3s


In [4]:
import torch.nn as nn
import numpy as np
from sklearn.model_selection import StratifiedKFold
import matplotlib.pyplot as plt 

In [5]:
import os
# Enable precise CUDA kernel error reporting
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
os.environ["TORCH_USE_CUDA_DSA"] = "1"
# Import PyTorch AFTER setting the environment variables
import torch

In [6]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)
print("Device:", device)
if not torch.cuda.is_available():
    raise RuntimeError("CUDA is unavailable; refusing to train on CPU.")
print("GPU:", torch.cuda.get_device_name(0))

Device: cuda
GPU: NVIDIA GeForce RTX 5060 Laptop GPU


## **Building the dataset**

In [7]:
from Data_pre_processing.data_cleaning import data_cleaning, split_train_test
import torch
from torch.utils.data import Dataset
from Data_pre_processing.Signal_treatement import split_signals_and_build

### **Getting the data**

La fonction upload_data retourn deux tuples : 
- Le premier tuple (training_x, training_y)
- Le deuxieme tuple (testing_x, testing_y)

donc pour acceder il suffit d'utiliser les indices 0 et 1, exemple : 

- training[0] donne le training_x, training[1] donne le training_y

In [8]:
from Data_pre_processing.Data_pipeline import upload_data

In [9]:
training, testing = upload_data(segment_duration=3) 

Normal: 522
Abnormal: 426


### **Building the dataset using PyTorch for training**

In [10]:
from Data_pre_processing.Build_dataset import PCGDataset

In [11]:
Training_data = PCGDataset(bags = training[0], labels = training[1])
Testing_data = PCGDataset(bags = testing[0], labels = testing[1])

## **Training**

- Pour l'entrainement on a le problème d'umblanced data, donc pour valider le modele, on vas utiliser le stratified cross validation.
- Le nombre de folds qu'on vas utiliser maintenant c'est 5.
- 1 pour abnormal, 0 pour normal.
- In this case we are going to priviluaze our folds to see the data distribution

### **Stratified cross validation**

In [12]:
skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

### **Training phase**

In [13]:
from Data_pre_processing.Build_dataset import build_data_loaders
from Models.MIL1D import Mil_classifier
from Training.Train_model import TrainModel

#### **Training and validation**

Ici je défini les Hyper parametres :

In [18]:
NUM_EPOCHS = 80
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
fold_results = []
MOMENTUM = 1e-4

Phase d'entrainement :

In [20]:
folds = []
pos_weight = torch.tensor([1.23], device=device)
for train_index, test_index in skf.split(
    np.zeros(len(training[1])),
    training[1],
):
    folds.append((train_index, test_index))

mlflow.set_experiment("PCG_MIL_3s_3Folds")
for fold, (train_idx, valid_idx) in enumerate(folds) : 
    print(f"training on the fold {fold}")
    print(f"*"*20, "INFORMATION ABOUT THE TRAINING", "*"*20)
    experiment_name_fold = f"{experiment_name}3Folds_fold{fold}"
    #MLflow new experiment
    numbers_of_zero = 0
    numbers_of_one = 0
    for index in train_idx :
      if training[1][index] == 1 :
        numbers_of_one+=1
      else : numbers_of_zero+=1
      
    print(f"      numbers of abnormal : {numbers_of_one}, numbers of normal : {numbers_of_zero}")

    torch.manual_seed(100 + fold)

    model = Mil_classifier().to(device=device)
    
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )

    checkpoint_path = CHECKPOINT_DIR / f"fold_{fold}.pt"

    start_epoch = 0

    if checkpoint_path.exists():
        checkpoint = torch.load(
            checkpoint_path,
            map_location=device,
            weights_only=True
        )

        if checkpoint["fold"] != fold:
            raise ValueError("Le checkpoint ne correspond pas au fold.")

        model.load_state_dict(
            checkpoint["model_state_dict"]
        )

        optimizer.load_state_dict(
            checkpoint["optimizer_state_dict"]
        )

        start_epoch = checkpoint["epoch"]

        print(
            f"Reprise du fold {fold} : "
            f"dernière époque terminée = {start_epoch}"
        )
    else:
        print(f"New training for the fold {fold}")

    train_loader, valid_loader = build_data_loaders(dataset=Training_data, train_index=train_idx, test_index=valid_idx)
    
    with mlflow.start_run(run_name=experiment_name_fold):
        
        mlflow.log_params({
            "optimizer" : "AdamW",
            "learning_rate" : LEARNING_RATE,
            "Momentum" : MOMENTUM,
            "Epochs" : NUM_EPOCHS,
            "weight_decay" : WEIGHT_DECAY,
            "pos_weight" : pos_weight,
            "early_stopping" : 20,
        })

        Training_model = TrainModel(
           model=model,
           criterion=criterion,
           optimizer=optimizer,
           train_loader=train_loader,
           device=device,
           valid_loader=valid_loader,
           start_epoch=start_epoch,
           number_epochs=NUM_EPOCHS,
           checkpoint_path=checkpoint_path,
           patience=20,
           min_delta = 0.001,
        )

        loss_res_train, loss_res_validation = Training_model.train_model(fold=fold)

        plt.plot(loss_res_train, label="Train loss")
        plt.plot(loss_res_validation, label="Validation loss")
        plt.xlabel("Epoch")
        plt.ylabel("Loss")
        plt.title(f"Training and Validation Loss for the fold {fold}")
        plt.legend()

        plt.savefig("loss_curve.png")
        plt.close()

        mlflow.log_artifact("loss_curve.png")
        
    

training on the fold 0
******************** INFORMATION ABOUT THE TRAINING ********************
      numbers of abnormal : 340, numbers of normal : 418
New training for the fold 0


c:\Users\aliai\Desktop\M2 I2A\Optimisation des modeles - Ouzar\Projet\venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


Fold : 0 | Train loss : 0.7031 | Train accuracy : 0.6464 | Validation loss : 0.7622 | Validation accuracy : 0.5474 | epoch : 1/80
checkpoint saved .. !
Fold : 0 | Train loss : 0.6619 | Train accuracy : 0.7137 | Validation loss : 0.7640 | Validation accuracy : 0.5895 | epoch : 2/80
Fold : 0 | Train loss : 0.6577 | Train accuracy : 0.7098 | Validation loss : 0.7575 | Validation accuracy : 0.6053 | epoch : 3/80
checkpoint saved .. !
Fold : 0 | Train loss : 0.6511 | Train accuracy : 0.7124 | Validation loss : 0.6865 | Validation accuracy : 0.7105 | epoch : 4/80
checkpoint saved .. !
Fold : 0 | Train loss : 0.6409 | Train accuracy : 0.7348 | Validation loss : 0.6603 | Validation accuracy : 0.7421 | epoch : 5/80
checkpoint saved .. !
Fold : 0 | Train loss : 0.6365 | Train accuracy : 0.7427 | Validation loss : 0.6288 | Validation accuracy : 0.7737 | epoch : 6/80
checkpoint saved .. !
Fold : 0 | Train loss : 0.6303 | Train accuracy : 0.7414 | Validation loss : 0.6160 | Validation accuracy : 0.

c:\Users\aliai\Desktop\M2 I2A\Optimisation des modeles - Ouzar\Projet\venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


Fold : 1 | Train loss : 0.6712 | Train accuracy : 0.7058 | Validation loss : 0.8101 | Validation accuracy : 0.5526 | epoch : 1/80
checkpoint saved .. !


c:\Users\aliai\Desktop\M2 I2A\Optimisation des modeles - Ouzar\Projet\venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


Fold : 1 | Train loss : 0.6787 | Train accuracy : 0.6834 | Validation loss : 0.8163 | Validation accuracy : 0.5526 | epoch : 2/80


c:\Users\aliai\Desktop\M2 I2A\Optimisation des modeles - Ouzar\Projet\venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


Fold : 1 | Train loss : 0.6737 | Train accuracy : 0.6900 | Validation loss : 0.7829 | Validation accuracy : 0.5526 | epoch : 3/80
checkpoint saved .. !
Fold : 1 | Train loss : 0.6494 | Train accuracy : 0.7243 | Validation loss : 0.7964 | Validation accuracy : 0.5684 | epoch : 4/80
Fold : 1 | Train loss : 0.6355 | Train accuracy : 0.7309 | Validation loss : 0.7937 | Validation accuracy : 0.5789 | epoch : 5/80
Fold : 1 | Train loss : 0.6253 | Train accuracy : 0.7388 | Validation loss : 0.7615 | Validation accuracy : 0.6158 | epoch : 6/80
checkpoint saved .. !
Fold : 1 | Train loss : 0.6179 | Train accuracy : 0.7454 | Validation loss : 0.7284 | Validation accuracy : 0.6263 | epoch : 7/80
checkpoint saved .. !
Fold : 1 | Train loss : 0.6177 | Train accuracy : 0.7414 | Validation loss : 0.7197 | Validation accuracy : 0.6316 | epoch : 8/80
checkpoint saved .. !
Fold : 1 | Train loss : 0.6139 | Train accuracy : 0.7546 | Validation loss : 0.7202 | Validation accuracy : 0.6316 | epoch : 9/80
Fo

KeyboardInterrupt: 